In [21]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.stats import norm

# ---------------- settings (change these, rerun) ----------------
RISK_FREE_RATE = 0.04       # assumed (not in the data; the forward below absorbs most of its effect)
STRIKE_WINDOW = 0.10        # keep strikes within +/-10% of the forward
MAX_REL_SPREAD = 0.25       # (ask-bid)/mid cap (quote-based days only)
MIN_GOOD_QUOTES = 100       # days with fewer usable bid/ask rows (zeroed quotes) are dropped
SERIES = "SPX"              # monthly series; SPXW is a different contract set, so don't mix them
FORWARD_STRIKE_BAND = 0.03  # strikes within +/-3% of spot are used to estimate the forward

# ---------------- load ----------------
raw = pd.read_csv("raw.csv", index_col=0).reset_index(drop=True)
raw["snapshot_time"] = pd.to_datetime(raw.snapshot_utc, utc=True)
raw["last_trade_time"] = pd.to_datetime(raw.lastTradeDate, utc=True)
raw["series"] = raw.contractSymbol.str.extract(r"^(SPXW?)")[0]

# Label each snapshot by the trading day its quotes belong to (Denver date; weekend -> Friday).
snapshot_time_denver = raw.snapshot_time.dt.tz_convert("America/Denver")
trading_day = snapshot_time_denver.dt.normalize().dt.tz_localize(None)
trading_day = trading_day.where(
    trading_day.dt.dayofweek < 5,
    trading_day - pd.to_timedelta((trading_day.dt.dayofweek - 4), unit="D"),
)
raw["trading_day"] = trading_day

options = raw[raw.series == SERIES].copy()
options["years_to_expiry"] = (pd.to_datetime(options.expiry) - options.trading_day).dt.days / 365.0

# ---------------- choose the price for each row ----------------
options["has_valid_quote"] = (options.bid > 0) & (options.bid < options.ask)

# Snapshots pulled after hours have bid/ask zeroed out, so no mid price exists. Drop those whole days.
valid_quotes_per_day = options.groupby("trading_day").has_valid_quote.sum()
dropped_days = valid_quotes_per_day[valid_quotes_per_day < MIN_GOOD_QUOTES].index
print("dropped days (bid/ask zeroed):", [f"{day:%a %m-%d}" for day in dropped_days])
options = options[~options.trading_day.isin(dropped_days)].copy()

options["mid_price"] = (options.bid + options.ask) / 2
options["rel_spread"] = (options.ask - options.bid) / options.mid_price
options["usable"] = options.has_valid_quote & (options.rel_spread < MAX_REL_SPREAD)
usable_rows = options["usable"]

# ---------------- forward from call/put pairs (per day, per expiry) ----------------
def estimate_forward(chain):
    """Median of  F = K + e^{rT} (C - P)  over strikes near spot that have both a call and a put."""
    chain = chain[usable_rows.loc[chain.index]
                  & ((chain.strike / chain.spot - 1).abs() < FORWARD_STRIKE_BAND)]
    if chain.empty:
        return np.nan
    prices_by_type = chain.pivot_table(index="strike", columns="type", values="mid_price")
    if not {"C", "P"} <= set(prices_by_type.columns):
        return np.nan
    prices_by_type = prices_by_type.dropna()
    if prices_by_type.empty:
        return np.nan
    years_to_expiry = chain["years_to_expiry"].iloc[0]
    estimates = (prices_by_type.index.values
                 + np.exp(RISK_FREE_RATE * years_to_expiry) * (prices_by_type["C"] - prices_by_type["P"]))
    return float(np.median(estimates))

forward_by_group = (options.groupby(["trading_day", "expiry"])
                    .apply(estimate_forward, include_groups=False)
                    .rename("forward"))
options = options.join(forward_by_group, on=["trading_day", "expiry"])

# ---------------- clean ----------------
clean = options[options.usable & options.forward.notna()]
clean = clean[(clean.strike / clean.forward - 1).abs() < STRIKE_WINDOW]
clean = clean[((clean.type == "C") & (clean.strike >= clean.forward))
              | ((clean.type == "P") & (clean.strike < clean.forward))]   # out-of-the-money side only

# ---------------- implied vol (invert Black-Scholes on the price) ----------------
def bs_paper(asset_price, strike, rate, volatility, years_to_expiry, option_type):
    """The paper's closed form: eqs (40), (41), (43).  asset_price = S, strike = E, years_to_expiry = T - t."""
    sigma_sqrt_t = volatility * np.sqrt(years_to_expiry)
    d1 = (np.log(asset_price / strike) + (rate + 0.5 * volatility**2) * years_to_expiry) / sigma_sqrt_t  # eq (41), upper sign
    d2 = (np.log(asset_price / strike) + (rate - 0.5 * volatility**2) * years_to_expiry) / sigma_sqrt_t  # eq (41), lower sign
    discounted_strike = strike * np.exp(-rate * years_to_expiry)
    if option_type == "C":
        return asset_price * norm.cdf(d1) - discounted_strike * norm.cdf(d2)      # eq (40)
    return discounted_strike * norm.cdf(-d2) - asset_price * norm.cdf(-d1)        # eq (43)

def implied_volatility(row):
    # The paper assumes no dividends, so the asset price must be replaced by an effective price that
    # carries the dividend effect. The parity forward gives it: effective = forward * e^{-rT} (= S when q = 0).
    effective_asset_price = row.forward * np.exp(-RISK_FREE_RATE * row["years_to_expiry"])

    def price_error(volatility):
        model_price = bs_paper(effective_asset_price, row.strike, RISK_FREE_RATE,
                               volatility, row["years_to_expiry"], row.type)
        return model_price - row.mid_price

    try:
        return brentq(price_error, 1e-4, 3.0)
    except ValueError:
        return np.nan

clean = clean.copy()
clean["implied_vol"] = clean.apply(implied_volatility, axis=1)
clean = clean.dropna(subset=["implied_vol"])

# ---------------- report ----------------
print("rows surviving per day/expiry:")
print(clean.groupby(["trading_day", "expiry"]).size().unstack())
print("\nforward vs spot:")
print(options.groupby(["trading_day", "expiry"])[["spot", "forward"]].first())
clean[["trading_day", "expiry", "type", "strike", "forward", "years_to_expiry",
       "bid", "ask", "mid_price", "implied_vol"]].to_csv("iv_clean.csv", index=False)

# ---------------- plot ----------------
# Goal: one panel per expiry date, with one set of points per trading day, so you can see
# (a) whether the points form a flat line (constant-sigma model) or a curve (the "smile"), and
# (b) how the curve moves from day to day.

# Two side-by-side panels (December on the left, January on the right).
# sharey=True gives both the same vertical scale so the panels can be compared directly.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

# One color per trading day, taken from the viridis colormap (dark purple -> green),
# so the earliest day is at one end of the scale and the latest at the other.
colors = plt.cm.viridis(np.linspace(0, 0.9, clean.trading_day.nunique()))

# groupby("expiry") hands us one chunk of rows per expiry date, in date order;
# zip pairs each chunk with the next panel in `axes`.
for axis, (expiry, expiry_rows) in zip(axes, clean.groupby("expiry")):

    # Inside each panel, split that expiry's rows by trading day and draw each day separately.
    for day, day_rows in expiry_rows.groupby("trading_day"):

        # x-axis: strike divided by the forward. Using this ratio instead of the raw strike
        # puts all days on a common scale even though the index level moved between days.
        # A value of 1.0 means "strike equals the forward" (the center of the smile).
        strike_over_forward = day_rows.strike / day_rows.forward

        # y-axis: the implied volatility from the inversion. "o" draws markers only (no connecting
        # lines), so gaps in the strikes stay visible instead of being bridged by a line.
        axis.plot(strike_over_forward, day_rows.implied_vol, "o",markerfacecolor='none' , ms=3, label=day.strftime("%a %m-%d"))

    axis.set_title(f"Expiry Date: {expiry}")
    axis.set_xlabel("Ratio of Strike to Forward")
    axis.legend(fontsize=8)       # legend entries are the trading days

axes[0].set_ylabel("Implied Volatility")  # shared y-axis, so only the left panel needs the label
plt.tight_layout()                 # keep titles and labels from overlapping
plt.savefig("volplots.png", dpi=150)  # also written to disk; the Agg backend means nothing pops up

dropped days (bid/ask zeroed): []
rows surviving per day/expiry:
expiry       2026-12-18  2027-01-15
trading_day                        
2026-11-16          132         152
2026-11-17          132         149
2026-11-19          135         137

forward vs spot:
                               spot      forward
trading_day expiry                              
2026-11-16  2026-12-18  7416.589844  7442.176484
            2027-01-15  7416.589844  7468.657183
2026-11-17  2026-12-18  7340.529785  7370.031022
            2027-01-15  7340.529785  7396.172628
2026-11-19  2026-12-18  7175.830078  7171.580972
            2027-01-15  7175.830078  7197.168172
